In [1]:
pip install WordCloud

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

$ Intro to Pandas

In [3]:
df= pd.read_csv("Euro.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'Euro.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df= pd.read_csv("ckd.csv")
#df.head()

# Min Max Normalization

# $x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$

In [ ]:
minmax_cols = ['bp','pc']

df_minmax = df.copy()

for col in minmax_cols:
    df_minmax[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

#print(df_minmax)
print(minmax_cols)

# Z-Score

# $z = \frac{x - \mu}{\sigma}$

# - $x$ is the original value  
# - $\mu$ is the mean of the feature  
# - $\sigma$ is the standard deviation of the feature

# $$
\sigma = \sqrt{\frac{1}{N - 1} \sum_{i=1}^{N} (x_i - \bar{x})^2}
$$ 

# where  
# - $x_i$ is the $i^{th}$ data value  
# - $\mu$ is the population mean  
# - $\bar{x}$ is the sample mean  
# - $N$ is the number of observations  
# - $\sigma$ is the standard deviation

In [ ]:
df_zscore = df.copy()

for col in minmax_cols:
    df_zscore[col] = (df[col] - df[col].mean()) / df[col].std()

print(df_zscore)

# Decimal Scaling

# Binning

In [ ]:
bins = [0, 11, 13, 20]
labels = ['Low', 'Normal', 'High']

df['hemo_level'] = pd.cut(df['hemo'], bins=bins, labels=labels)
print(df[['hemo', 'hemo_level']])

In [ ]:
df['age'].max()

In [ ]:
bins = [0, 25, 55, 90]
labels = ['Young', 'Middleaged', 'Senior']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
print(df[['age', 'age_group']])

# indexing

In [ ]:
dinx=df.copy()
dinx01=df.copy()
dinx.head()

In [ ]:
# dinx[['sg']]


In [ ]:
dinx=df.copy()
dinx.set_index(['id','age'], inplace=True)
#df.reset_index(inplace=True)
dinx.head()

In [ ]:
dinx.set_index(['id','age'], inplace=True)

In [ ]:
dinx=df.copy()
dinx.set_index(['age', 'rbc'], inplace=True)
dinx.head(20)

In [ ]:
#dinx.set_index(['gender', 'patient_id'], inplace=True)
dinx.sort_index(inplace=True)
dinx.head(20)

# Label encoding

In [ ]:
df['classification_encoded'] = df['classification'].map({
    'ckd': 1,
    'notckd': 0
})
print(df)

# One_hot encoding

In [ ]:
df_onehot = pd.get_dummies(
    df,
    columns=['appet'],
    prefix='appet'
)

print(df_onehot)

# NLP

In [ ]:
text = """Patient has Chronic Kidney Disease (CKD).
Blood pressure is high, and creatinine level is 2.5 mg/dL!!!"""

In [ ]:
import re
#text01=  text.lower()
text01 = re.sub(r'\d+', '', text) 
#text01 = re.sub(r'[^\w\s]', '', text)
#text01 = re.sub(r'\s+', ' ', text)
print(text01)

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
tokens = word_tokenize(text01)
print(tokens)

In [ ]:

stop_words = set(stopwords.words('english'))

filtered_tokens = list(filter(lambda word: word not in stop_words, tokens))
print(filtered_tokens)

In [ ]:
text = "Patient ID: P004 admitted today"

pid = re.findall(r'P\d{3}', text)
print(pid)

# IMAGE AUGMENTATION

In [ ]:
import cv2
import numpy as np

# Read image
img = cv2.imread("0.jpg")

# Convert BGR → RGB for visualization
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
cv2.imshow("Original Image", img)
cv2.waitKey(0)

# Sampling

In [ ]:
srs_sample = df.sample(n=10)

#print(srs_sample.head())
srs_sample.count()

# Systematic Sampling

In [ ]:
N = len(df)
n = 13
k = N // n  
#print(N)
print(k)
#k=10
systematic_sample = df.iloc[::k]

print(systematic_sample.head())

len(systematic_sample)

# Stratified Sampling

In [ ]:
stratified_sample = (
    df.groupby('pc', group_keys=False)
      .apply(lambda x: x.sample(frac=1, random_state=42))
)

print(stratified_sample['pc'].value_counts())

In [ ]:
df.isnull().sum()

# Cluster Sampling

In [ ]:
selected_clusters = df['pc'].drop_duplicates().sample(n=4, random_state=42)


cluster_sample = df[df['pc'].isin(selected_clusters)]
#print (len(selected_clusters))
len(cluster_sample)
#cluster_sample.count()
#print(cluster_sample.head())

# Outlier and Anomaly

# Z Score

In [ ]:
import numpy as np

df = pd.read_csv("ckd.csv")

bp = df['bp']

z_scores = (bp - bp.mean()) / bp.std()

outliers = df[z_scores.abs() > 3]
print(outliers[['bp']])

# IQR Method

In [ ]:
Q1 = df['bp'].quantile(0.1)
Q3 = df['bp'].quantile(0.9)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['bp'] < lower) | (df['bp'] > upper)]
print(outliers[['bp']])

# Feature selection

# 1. correlation


$$
r_{xy} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})} {\sqrt {\sum_{i=1}^{n} (x_i - \bar{x})^2} {\sum_{i=1}^{n} y_i - \bar{y})^2}}
$$


| r value | Meaning                      |
| ------- | ---------------------------- |
| +1      | Perfect positive correlation |
| -1      | Perfect negative correlation |
| 0       | No linear correlation        |


In [ ]:
df[['bp', 'hemo', 'sg', 'age']].corr()

In [ ]:
corr = df.corr()
strong_corr = corr[abs(corr) > 0.8]
print(strong_corr)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df[['bp','hemo','sg','age']].corr(), annot=True, cmap='coolwarm')
plt.show()

# 2. chi-square test
categorical only

In [ ]:
table=pd.crosstab(df['pc'], df['classification'])
table

In [ ]:
from scipy.stats import chi2_contingency

#chi2 = chi2_contingency(table)
chi2, p, dof, expected = chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)

expected
#dof

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

df = 2
alpha = 0.95

critical_value = stats.chi2.ppf(alpha, df)
critical_value

# Wrapper Methods

# forward selection, backward elimination, recursive feature

# Embedded methods

# PCA

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

df=df.dropna()


In [ ]:
features = ['age','bp','sg','hemo']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
X_pca

In [ ]:

#print(pca.components_)

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(X_pca[:,0], X_pca[:,1], c=df['classification'].astype('category').cat.codes)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Visualization of CKD Data")
plt.show()

# ANOVA

In [ ]:
#Regression

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = df[['age']]              
y = df['sc']                 

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Slope:", model.coef_[0])

In [ ]:
model.predict([[55]])

In [ ]:
from sklearn.linear_model import LinearRegression

X = df[['age', 'bp', 'hemo', 'sg']]
y = df['sc']

model = LinearRegression()
model.fit(X, y)

print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

In [ ]:
model.predict([[55, 90, 12, 120]])

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Example text dataset
docs = [
    "Chronic kidney disease is dangerous",
    "Kidney patients need regular checkups",
    "Disease progression can be slowed"
]

# Create Bag of Words
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(docs)

# Convert to DataFrame
df_bow = pd.DataFrame(bow_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df_bow)


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all text
text ="Chronic kidney disease is dangerous, Kidney patients need regular checkups, Disease progression can be slowed"

# Create Word Cloud
wordcloud = WordCloud(
    font_path='C:/Windows/Fonts/arial.ttf', 
    width=800, 
    height=400, 
    background_color='white'
).generate(text)

# Display
plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

In [ ]:
import cv2
import matplotlib.pyplot as plt

# ----------------------------
# Load Image
# ----------------------------
img = cv2.imread("19.jpg")       # Original image
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert for Matplotlib

plt.figure(figsize=(6,6))
plt.imshow(img_rgb)
plt.title("Original Image")
plt.axis('off')
plt.show()

# ----------------------------
# 1. Edge Detection (Canny)
# ----------------------------
edges = cv2.Canny(img, threshold1=100, threshold2=200)

plt.figure(figsize=(6,6))
plt.imshow(edges, cmap='gray')
plt.title("Canny Edge Detection")
plt.axis('off')
plt.show()

# ----------------------------
# 2. Sobel Detection
# ----------------------------

img = cv2.imread("19.jpg")
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
sobel_x = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(np.abs(sobel_x), cmap='gray')
plt.title("Sobel X (Vertical Edges)")
plt.axis('off')

# ----------------------------
# 3. Contour Detection
# ----------------------------

contours, hierarchy = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)


contour_img = img_rgb.copy()
cv2.drawContours(contour_img, contours, -1, (255, 0, 0), 2)

plt.figure(figsize=(6,6))
plt.imshow(contour_img)
plt.title("Contour Detection")
plt.axis('off')
plt.show()

# ----------------------------
# 4 Contour Analysis
# ----------------------------
# print("Total Contours Found:", len(contours))

# for i, cnt in enumerate(contours[:5], start=1):   # show first 5 contours
#     area = cv2.contourArea(cnt)
#     perimeter = cv2.arcLength(cnt, True)
    
#     print(f"\nContour #{i}")
#     print("Area:", area)
#     print("Perimeter:", perimeter)
